# START : Liz Choi

# Objective
The goal of this notebook is to retrain and evaluate a **Random Forest Regressor** for predicting residential property closing prices.
We use the newly engineered features (e.g., Bed_Bath_Ratio, Property_Age, Months_From_Dec_2025, Sqft_Per_Bedroom, Lot_Utilization) to improve predictive performance.

## Import Libraries and Load Data
We load the feature-engineered training and test datasets saved from the previous feature engineering step.

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

train = pd.read_csv("train_cleaned_fe.csv")
test  = pd.read_csv("test_cleaned_fe.csv")

## Log Transformation of Target Variable
We model the **log-transformed ClosePrice** to reduce skew and make errors more stable across price ranges.

In [2]:
train["LogClosePrice"] = np.log(train["ClosePrice"])
test["LogClosePrice"]  = np.log(test["ClosePrice"])

## Select Features for Random Forest
We define a numeric feature set (same style as the Decision Tree notebook).
All selected features are converted to numeric format.

In [3]:
rf_features = [
    "LivingArea",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "PPSF",
    "Bed_Bath_Ratio",
    "Property_Age",
    "Months_From_Dec_2025",
    "Sqft_Per_Bedroom",
    "Lot_Utilization"
]

for col in rf_features:
    train[col] = pd.to_numeric(train[col], errors="coerce")
    test[col]  = pd.to_numeric(test[col], errors="coerce")

## Remove NaN / inf
Random forests cannot handle NaN/inf values directly.
We replace inf with NaN and drop rows that are missing required features or the target.

In [4]:
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

train = train.dropna(subset=rf_features + ["LogClosePrice"])
test  = test.dropna(subset=rf_features + ["LogClosePrice"])

## Prepare Feature Matrices and Target Variables
- **X_train / X_test**: input feature matrices
- **y_train / y_test**: log-transformed target (`LogClosePrice`)
Features are cast to **float32** to improve efficiency.

In [5]:
X_train = train[rf_features].astype(np.float32)
X_test  = test[rf_features].astype(np.float32)

y_train = train["LogClosePrice"]
y_test  = test["LogClosePrice"]

## Train Random Forest Regressor
A Random Forest is an ensemble of many decision trees.
Key parameters control complexity and help reduce overfitting:
- `n_estimators`: number of trees
- `max_depth`: max tree depth
- `min_samples_leaf`: minimum samples per leaf

In [6]:
rf_model = RandomForestRegressor(
    n_estimators=600,
    max_depth=18,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=18, min_samples_leaf=20, n_estimators=600,
                      n_jobs=-1, random_state=42)

## Generate Predictions
We generate predictions on both train and test sets.
Predictions are in **log scale** and later converted back to dollar scale for error metrics.

In [7]:
train_pred = rf_model.predict(X_train)
test_pred  = rf_model.predict(X_test)

## Model Performance Evaluation
We report:
- **R²** on log scale
- **RMSE** on dollar scale
- **MAPE** and **MdAPE** on dollar scale

In [8]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_percentage_error, root_mean_squared_error

train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)

y_train_d = np.exp(y_train)
y_test_d  = np.exp(y_test)

train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

train_rmse = root_mean_squared_error(y_train_d, train_pred_d)
test_rmse  = root_mean_squared_error(y_test_d, test_pred_d)

train_mape = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape  = mean_absolute_percentage_error(y_test_d, test_pred_d) * 100

train_mdape = np.median(np.abs((y_train_d - train_pred_d) / y_train_d)) * 100
test_mdape  = np.median(np.abs((y_test_d - test_pred_d) / y_test_d)) * 100

print("Random Forest Performance")
print(f"Train R2 (log): {train_r2:.4f} | Test R2 (log): {test_r2:.4f}")
print(f"Train RMSE ($): {train_rmse:,.2f} | Test RMSE ($): {test_rmse:,.2f}")
print(f"Train MAPE (%): {train_mape:.2f} | Test MAPE (%): {test_mape:.2f}")
print(f"Train MdAPE(%): {train_mdape:.2f} | Test MdAPE(%): {test_mdape:.2f}")

Random Forest Performance
Train R2 (log): 0.9991 | Test R2 (log): 0.9985
Train RMSE ($): 64,397.26 | Test RMSE ($): 95,171.73
Train MAPE (%): 0.55 | Test MAPE (%): 0.63
Train MdAPE(%): 0.26 | Test MdAPE(%): 0.30


## Feature Importance
Random Forest provides feature importance scores which help interpret which engineered features contribute most.

In [9]:
importances = pd.DataFrame({
    "Feature": rf_features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importances

,Feature,Importance
3,PPSF,5.907522e-01
0,LivingArea,4.092433e-01
7,Sqft_Per_Bedroom,3.281842e-06
8,Lot_Utilization,4.176657e-07
5,Property_Age,2.280924e-07
1,BathroomsTotalInteger,2.134192e-07
2,BedroomsTotal,1.762072e-07
6,Months_From_Dec_2025,1.379054e-07
4,Bed_Bath_Ratio,4.404224e-08


# END : Liz Choi

# START: Steven Shi

## Old Features vs. New Features
Old features:

* All features (one-hot encoded)

New Features:

* LivingArea,
* BathroomsTotalInteger,
*    BedroomsTotal,
*    PPSF,
*    Bed_Bath_Ratio,
*   Property_Age,
*    Months_From_Dec_2025,
*    Sqft_Per_Bedroom,
*    Lot_Utilization.


In [ ]:
# Calculating the R^2 Score after prediction has been converted back to original $ units
# Actual values in original scale
y_train_d = train['ClosePrice']
y_test_d  = test['ClosePrice']

# Convert predictions back to original scale
train_pred_d = np.expm1(train_pred)
test_pred_d  = np.expm1(test_pred)

# Calculate R² in original dollar scale
train_r2_d = r2_score(y_train_d, train_pred_d)
test_r2_d  = r2_score(y_test_d, test_pred_d)

print(f"Train R^2 (Original $): {train_r2_d:.4f}")
print(f"Test R^2 (Original $): {test_r2_d:.4f}")

Train R^2 (Original $): 0.9954
Test R^2 (Original $): 0.9896


In [ ]:
# draw the evaluation table to compare the models' performance.

import pandas as pd
import numpy as np

data = {
    "Metric": [
        "R² (log scale)",
        "R² (original $)",
        "RMSE ($)",
        "MAPE (%)",
        "MdAPE (%)"
    ],

    "Baseline Train": [0.762, 0.632, np.nan, 22.16, 15.91],
    "Baseline Test": [0.727, 0.590, np.nan, 23.90, 17.08],

    "RF Train": [0.982088, 0.975642, np.nan, 5.719985, 3.825515],
    "RF Test": [0.865883, 0.812075, np.nan, 15.979177, 10.428067],

    "RF (New) Train": [0.9991, 0.9954, 64397.26, 0.55, 0.26],
    "RF (New) Test": [0.9985, 0.9896, 95171.73, 0.63, 0.30],
}

df = pd.DataFrame(data)

# Round numbers for better presentation
df = df.round(4)

# Set Metric as index for cleaner display
df = df.set_index("Metric")

# Display table
df

,Baseline Train,Baseline Test,RF Train,RF Test,RF (New) Train,RF (New) Test
Metric,,,,,,
R² (log scale),0.762,0.727,0.9821,0.8659,0.9991,0.9985
R² (original $),0.632,0.590,0.9756,0.8121,0.9954,0.9896
RMSE ($),NaN,NaN,NaN,NaN,64397.2600,95171.7300
MAPE (%),22.160,23.900,5.7200,15.9792,0.5500,0.6300
MdAPE (%),15.910,17.080,3.8255,10.4281,0.2600,0.3000


The new Random Forest model clearly outperforms the original Random Forest model on both training and test sets across all metrics. There is a great improvement in test R^2 from 0.8659 to 0.9985, explaining almost all variance in the log-transformed house prices. The test R^2 in original scale also increase largely, showing that the new model predicts actual house prices much more accurately.

The MAPE and MdAPE indicate a dramatic reduction in prediction error, meaning the new model's predictions are very close to the true prices for most observations.

## Reason of the Improvement

In [11]:
# Feature Importance in float format
pd.set_option('display.float_format', '{:.8f}'.format)
importances

,Feature,Importance
3,PPSF,0.59075219
0,LivingArea,0.40924331
7,Sqft_Per_Bedroom,0.00000328
8,Lot_Utilization,0.00000042
5,Property_Age,0.00000023
1,BathroomsTotalInteger,0.00000021
2,BedroomsTotal,0.00000018
6,Months_From_Dec_2025,0.00000014
4,Bed_Bath_Ratio,0.00000004


`PPSF` contributes about 59% of the model's importance. This means the model is essentially using price (Price Per Square Foot) to predict price, which is a form of data leakage.

`LivingArea` contributes about 41%.

Because PPSF * LivingArea ≈ Price, the two variables together almost reconstruct the target variable directly.



All other features contribute almost nothing

# END Steven Shi